# Session 4: scikit-learn, preprocessing and a first linear regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-hossein/hands-on-ml-biotech/blob/main/03_ObjectsAndSklearn/02_Sklearn.ipynb)

In the last session, we wrote `MeanPredictor`, a class with `fit` and `predict`. **scikit-learn** (imported as `sklearn`) is a library of models that all follow that same pattern. Today we learn the pattern once, and use it for three purposes:

- splitting data and measuring how wrong a model is,
- preparing data (**preprocessing**),
- fitting a first **linear regression**, and the linear algebra behind it: how `fit` actually finds the line.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

DATA = "../data/" if Path("../data").exists() else "https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/data/"

weather = pd.read_csv(DATA + "weather.csv")
weather["time"] = pd.to_datetime(weather["time"].astype(str), format="%Y%m%d%H")
weather["LUZ_wind_peak_in5h"] = weather["LUZ_wind_peak"].shift(-5) # the target: wind peak in Luzern 5 hours later
weather = weather.dropna() # the last 5 hours have no target

bikes = pd.read_csv(DATA + "bikes.csv")

print("weather", weather.shape, "\nbikes", bikes.shape)

weather (25213, 58) 
bikes (17379, 13)


## 1. Every scikit-learn model looks the same

| Step | Code | What happens |
| --- | --- | --- |
| 1. Choose a class | `from sklearn.linear_model import LinearRegression` | |
| 2. Create an object | `model = LinearRegression()` | setting hyperparameters |
| 3. Learn from data | `model.fit(X, y)` | what was learned is stored on the object, in attributes ending with `_` |
| 4. Use it | `model.predict(X_new)` | |

Two conventions for the data:

- **`X`** is a table with one row per sample and one column per feature: shape `(n, p)` (always 2D).
- **`y`** is the target: one value per sample, shape `(n,)`.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell2_image.png)

![image-2.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell2_image-2.png)

*from Hands-On Machine Learning with Scikit-Learn and PyTorch*

In [2]:
X = weather[["LUZ_pressure"]]
y = weather["LUZ_wind_peak_in5h"]

print(X.shape, y.shape)

(25213, 1) (25213,)


The simplest model in scikit-learn is `DummyRegressor`: it ignores `X` and always predicts the mean of `y`. That is exactly our `MeanPredictor`.

In [3]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X, y)
dummy.predict(X[:3])

array([13.67223258, 13.67223258, 13.67223258])

## 2. Training set and test set

A model that is judged on the data it learned from can look much better than it is. So we hide part of the data during `fit` and use it only to measure the error:

- **training set**: used by `fit`
- **test set**: used only for evaluation

`train_test_split` picks the test rows at random. `random_state` fixes the random choice, so that everyone gets the same split.

In [ ]:
# train_test_split with test_size=0.2 and random_state=0, then print the four shapes

**Be Careful**: Our weather data is a time series: one hour looks a lot like the hour before. Shuffling time-series data can lead to *data leakage*.

For data like this, split by time: learn from the past, test on the future.

In [4]:
is_train = weather["time"] < "2017-01-01"

X_train, y_train = X[is_train], y[is_train]
X_test, y_test = X[~is_train], y[~is_train]

print(len(X_train), "training hours,", len(X_test), "test hours")

16739 training hours, 8474 test hours


## 3. How wrong is a model?

For regression, the usual measure is the **mean squared error** between the true values $y_i$ and the predictions $\hat{y}_i$:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Its square root, the **RMSE**, has the same unit as the target (here km/h), so it is easier to read.

Before any real model, measure the error of the dummy model as a **baseline**.

In [ ]:
# fit a DummyRegressor on the training set (call it dummy_model) and compute its RMSE on the test set

### ✏️ Exercise 3.2: Two baselines for five-hour wind forecasting

Is the mean really the best we can do without a model? Another obvious guess: *the wind in 5 hours will be the wind now* (**persistence**).

Here we use the whole dataset, as on the exercise sheet, so the numbers differ slightly from the train/test numbers above.

1. Load the raw weather data and build the five-hour-ahead target:
```python
raw = pd.read_csv(DATA + "weather.csv")
y5 = raw["LUZ_wind_peak"][5:].values
```
2. **Mean baseline.** Predict the mean of `y5` for every hour. Compute the MSE.
3. **Persistence baseline.** Predict the wind peak observed now: `raw["LUZ_wind_peak"][:-5].values`. Compute the MSE.
4. Which baseline is better, and by how much?
5. Persistence looks like the obvious choice, yet at five hours it loses. Why? At which horizon would you expect it to win? (Try 1, 3, 5, 12 and 24 hours.)
6. Compare the MSE of the mean baseline with `y5.var()`. Why are they identical?

In [ ]:
# YOUR CODE HERE

## 4. A first linear regression

Linear regression with one feature predicts with a straight line:

$$\hat{y} = \theta_0 + \theta_1 x$$

In [ ]:
# create a LinearRegression (call it linear_model), fit it on the training set, print coef_ and intercept_

- $\theta_1$ is the **slope**, stored in `coef_`
- $\theta_0$ is the **intercept**, stored in `intercept_`

In [ ]:
x_line = pd.DataFrame({"LUZ_pressure": [930, 990]}) # why?

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_test["LUZ_pressure"], y_test, s=5, alpha=0.15, label="test data (2017)")
ax.plot(x_line["LUZ_pressure"], linear_model.predict(x_line), color="red", label="linear regression")
ax.axhline(dummy_model.constant_.item(), color="black", linestyle="--", label="baseline (mean)")
ax.set_xlabel("pressure in Luzern (hPa)")
ax.set_ylabel("wind peak 5 hours later (km/h)")
ax.set_title("One feature: only slightly better than the mean")
ax.legend()
plt.show()

In [ ]:
# RMSE of the linear model on the test set. Does it beat the baseline?

One feature helps a little. With several features the model is no longer a line but the idea is the same, one coefficient per feature:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_p x_p$$

For scikit-learn nothing changes: `X` just gets more columns.

In [ ]:
luzern = ["LUZ_wind_peak", "LUZ_pressure", "LUZ_temperature", "LUZ_precipitation",
          "LUZ_sunshine_duration", "LUZ_wind_mean", "LUZ_wind_direction"]

X_train, X_test = weather.loc[is_train, luzern], weather.loc[~is_train, luzern]

In [ ]:
# fit a LinearRegression on the 7 Luzern features (call it linear_luzern), print its test RMSE and its coefficients

In [ ]:
pd.Series(linear_luzern.coef_, index=luzern) 

## 5. What does `fit` compute? The linear algebra behind linear regression

This section follows the lecture: the same formulas, now with real numbers so you can check each step.

### 5.1 The model as a matrix product

Stack the $N$ training samples as rows of a matrix $\mathbf{X}$ ($N \times D$) and the targets in a vector $\mathbf{y}$ ($N$). The model predicts all samples at once:

$$\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}, \qquad \hat{y}_n = \mathbf{x}_n^\top \mathbf{w} = w_1 x_{n1} + \dots + w_D x_{nD}$$

- Where is the intercept $\theta_0$? Add a **column of ones** to $\mathbf{X}$
- In NumPy, a matrix product is `@`.

In [ ]:
X_small = np.array([[1, 2.0, 0.5],
                    [1, 1.0, 1.5],
                    [1, 3.0, 2.0]])
w_small = np.array([10.0, 2.0, -1.0]) 

print(X_small @ w_small)

### 5.2 The loss in matrix form

$$\mathcal{L}(\mathbf{w}) = \frac{1}{2N}\sum_{n=1}^{N}\left(y_n - \mathbf{x}_n^\top\mathbf{w}\right)^2 = \frac{1}{2N}(\mathbf{y} - \mathbf{X}\mathbf{w})^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$$

The vector $\mathbf{e} = \mathbf{y} - \mathbf{X}\mathbf{w}$ holds the errors.


### 5.3 Setting the gradient to zero: the normal equations

Expand the loss (using $(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top\mathbf{A}^\top$, and that a number equals its own transpose):

$$2N\,\mathcal{L}(\mathbf{w}) = \mathbf{y}^\top\mathbf{y} - 2\,\mathbf{w}^\top\mathbf{X}^\top\mathbf{y} + \mathbf{w}^\top\mathbf{X}^\top\mathbf{X}\,\mathbf{w}$$

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/04_SklearnAndLinearRegression/images/01_LinearRegression_cell26_image.png)

Thus:

$$\nabla\mathcal{L}(\mathbf{w}) = -\frac{1}{N}\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\mathbf{w})$$

At the minimum the gradient is zero, which gives the **normal equations**:

$$\mathbf{X}^\top\mathbf{X}\,\mathbf{w}^\star = \mathbf{X}^\top\mathbf{y}$$

$\mathbf{X}^\top\mathbf{X}$ is the **Gram matrix**, $D \times D$ (here $8 \times 8$). If it is invertible, $\mathbf{w}^\star = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$.

### 5.4 Two views of the same solution

**Convexity.** The loss is a quadratic function of $\mathbf{w}$, and $\mathbf{w}^\top\mathbf{X}^\top\mathbf{X}\mathbf{w} = \|\mathbf{X}\mathbf{w}\|^2 \ge 0$, so it is a bowl with no other dips: the point where the gradient is zero is the global minimum. With one parameter, the bowl is a parabola:

In [ ]:
slopes = np.linspace(w_star[2] - 0.02, w_star[2] + 0.02, 101)
losses = [loss(np.r_[w_star[:2], s, w_star[3:]]) for s in slopes]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(slopes, losses)
ax.axvline(w_star[2], color="red", linestyle="--", label="normal equation solution")
ax.set_xlabel("weight of LUZ_pressure")
ax.set_ylabel("training loss")
ax.set_title("The loss is a parabola: one minimum, no traps")
ax.legend()
plt.show()

**Probability.** The log-likelihood of the data is

$$\log p(\mathbf{y} \mid \mathbf{X}, \mathbf{w}) = -\frac{1}{2\sigma^2}\sum_{n}(y_n - \mathbf{x}_n^\top\mathbf{w})^2 + \text{const}$$

The only part that depends on $\mathbf{w}$ is minus the sum of squared errors. So **maximising the likelihood is the same as minimising the MSE**: both machines find the same $\mathbf{w}^\star$. Squared error is not an arbitrary choice: it is what Gaussian noise implies.

### ✏️ The mean is a linear regression

Take a model with only the intercept: $\mathbf{X}$ is a single column of ones. Write out $\mathbf{X}^\top\mathbf{X}$ and $\mathbf{X}^\top\mathbf{y}$ and solve the normal equation on paper. What is $w^\star$? Which model from this notebook is it? Check your answer in code.

*Your answer:*

In [ ]:
# check: solve the normal equation with a single column of ones

### ✏️ Exercise 4.1: Which statements are correct? (exercise sheet 3, exercise 1)

Data are generated by $y = 3x + 1 + \varepsilon$, with $\varepsilon \sim \mathcal{N}(0, \sigma^2)$. The training and test data are

$$D = ((0, 1), (2, 9)), \qquad D_\text{test} = ((0, 0), (3, 20))$$

We use the model family $f(x) = \theta_0 + \theta_1 x^2$ and the absolute-error loss $L(y, \hat{y}) = |y - \hat{y}|$. Which statements are correct? (Pen and paper.)

1. The training loss is smallest for $\hat{\theta}_0 = 1$ and $\hat{\theta}_1 = 2$.
2. For $f(x) = 1 + 2x^2$, the conditional test loss at $x = 0$ is 1. *Hint:* $E|\varepsilon| = \sigma\sqrt{2/\pi}$.
3. For $f(x) = 1 + 2x^2$, the average loss on the given test set is 1.

*Your answers:*

## 6. Preprocessing: the main steps

Models need data that is **numerical**, **complete** and often **on comparable scales**. Real tables are none of these. scikit-learn's tools for fixing this are called **transformers**. They follow the same pattern, with `transform` instead of `predict`:

- `fit(X)` learns what is needed from the data (a mean, the list of categories, ...)
- `transform(X)` applies it and returns the new data
- `fit_transform(X)` does both in one call

A small table with all three problems:

In [ ]:
samples = pd.DataFrame({
    "age": [34, 51, np.nan, 45, 62, 29],
    "expression": [1200.0, 15.0, 300.0, np.nan, 8000.0, 50.0],
    "tissue": ["liver", "brain", "liver", "blood", "brain", "blood"],
})
samples

### Missing values: `SimpleImputer`

Last session we dropped rows with `dropna` or filled them with `fillna`. `SimpleImputer` fills each column with a statistic of that column, and remembers the value so that new data is filled the same way.

In [ ]:
# SimpleImputer(strategy="median") on age and expression. Look at statistics_

### Scaling: `StandardScaler`

Standardisation subtracts the mean of a column and divides by its standard deviation:

$$z = \frac{x - \text{mean}}{\text{std}}$$

Afterwards every column has mean 0 and standard deviation 1, whatever its original unit.

In [ ]:
# StandardScaler on the filled array. Look at mean_ and scale_

A transformer is fitted on the **training set only**, and then used to transform both the training set and the test set.

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test) 
```

If the test set takes part in `fit`, information about it leaks into the model, and the test error is no longer honest. This is another example of **data leakage**.

### Categories: `OneHotEncoder`

A model cannot multiply "liver" by a coefficient. Numbering the categories (blood = 0, brain = 1, liver = 2) would be wrong too: it claims that liver is "twice" brain. **One-hot encoding** makes one 0/1 column per category instead.

In [ ]:
# OneHotEncoder(sparse_output=False) on tissue. Look at get_feature_names_out()

### Back to the weather

Scale the 7 Luzern features (fit on the training set only) and fit the model again. Compare the error and the coefficients.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # fit on the training set
X_test_scaled = scaler.transform(X_test) # only transform the test set

linear_scaled = LinearRegression()
linear_scaled.fit(X_train_scaled, y_train)

print("test RMSE, raw features:   ", round(rmse_luzern, 2))
print("test RMSE, scaled features:", round(root_mean_squared_error(y_test, linear_scaled.predict(X_test_scaled)), 2))

pd.DataFrame({"raw": linear_luzern.coef_, "scaled": linear_scaled.coef_}, index=luzern).round(2)

- **The error did not change.** For linear regression, scaling does not change the predictions (it does change how well-conditioned $\mathbf{X}^\top\mathbf{X}$ is).
- **The coefficients became comparable.** On the raw scale precipitation looks strongest; on a common scale the current wind peak clearly is.

### Plot first, then preprocess

So far we chose the preprocessing from the *type* of a column: numbers get scaled, categories get one-hot encoded. But two numerical columns can need very different treatment, and a table of numbers hides this. A histogram shows it at a glance.

We plot the **training set only**: choosing the preprocessing is part of building the model, so the test set should not influence it.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell31_image.png)

Read more: 
- https://medium.com/@myonchev99/log-transformations-and-its-implications-for-linear-regression-f4570de0d69d
- https://scikit-learn.org/stable/auto_examples/compose/plot_transformed_target.html

## 7. ✏️ Exercise 4.2: Deterministic, yet it looks noisy

In the scatter plot of pressure against wind, one pressure value came with many different wind peaks. Is that "noise"? This exercise shows how noise can appear when we only look at some of the predictors.

The time a feather takes to fall one metre depends on its fluffiness $f \in [1, 4)$, its shape $x \in \{s, e\}$ (spherical or elongated) and the wind speed $w \in [0, 5)$ m/s:

$$g(f, x, w) = 0.1f^2 + 0.2w + 2\,I(x = s) + 0.5\,I(x = e)$$

1. Generate a dataset of 500 experiments with random feathers and conditions (`rng.uniform`, `rng.choice`).
2. Scatter plot: fluffiness on the x axis, falling time on the y axis.
3. Explain why the time *looks* random given the fluffiness, although $g$ has no random term. Then colour the points by shape: what do you see?
4. *(Optional, extra credit.)* Explain question 3 mathematically. Start from $p(t \mid f, x, w) = \delta(t - g(f, x, w))$, marginalise over the other variables, and compute $p(t \mid f = 2)$. Is it a delta distribution?

In [ ]:
# YOUR CODE HERE

## Homework

1. **Bike rentals** below (Optional).
2. **Exercise 3.1** ask an LLM to describe the bike rental dataset (columns, ranges, number of rows, target). Check three of its claims against the data, with one line of code each. Would you have noticed its mistakes without checking?
3. **Exercise 3.3** in the spam data (`pd.read_csv(DATA + "spam.csv", nrows=6000).dropna()`), what accuracy does "always ham" reach? Is a 90% accurate filter good here? Would it be good in a mailbox where only 3 in 100 emails are spam? Two filters have the same accuracy: one lets spam through, the other deletes real emails. Are they equally good, and what would you report instead of accuracy?
4. *(Extra credit)* Exercise 4, question 4.

### ✏️ Bike rentals (Optional)

1. Split `bikes` into a training and a test set with `train_test_split` (`test_size=0.2`, `random_state=0`). You can pass the whole DataFrame: `train, test = train_test_split(bikes, ...)`.
2. Compute the baseline RMSE on the test set with a `DummyRegressor`.
3. Fit a `LinearRegression` that uses only `temp`, and compute its test RMSE. What does the slope mean?

In [ ]:
# YOUR CODE HERE

*Homework answers (exercise sheet 2, exercise 3):*